In [4]:
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model="gpt-5-mini")

In [3]:
messages = [
    {"role": "system", "content": "You are a genZ poet."},
    {"role":"user", "content": "Write a poem about the beauty of nature"}
]

response = llm.invoke(messages)
print(response.content)

sunlight scrolls across my wrist
like a golden DM — casual, insistent.
grass whispers in a language my thumbs almost know,
each blade a tiny signal, green and urgent.

there's honesty in mud:
it sticks, it cools, it holds the memory of rain.
I kneel. the earth doesn't judge
for living in shoes that carry city dust.

a creek posts stories in ripples,
no captions, just movement — honest virality.
stones braid themselves into quiet timelines,
moss is the softest kind of bookmark.

a crow caws and the sky blushes,
clouds rearrange like mood filters.
a sunflower is loud in a way words are not:
all the light concentrated into a single, stubborn yes.

even the weeds have dignity,
flowers in unpaid internships, blooming anyway.
the moon shows up like she always does —
no RSVP, no schedule, just steady: luminous, patient.

I want to screenshot this moment but the pixels
can't capture the taste of wind, the small earthquake
of a heart learning how to be small beneath big things.
my phone vibrate

In [7]:
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a genZ poet."),
        ("user", "Write a poem about the {topic}.")
    ]
)
mountains_prompt = my_template.invoke({"topic": "beauty of mountains"})
mountains_prompt

ChatPromptValue(messages=[SystemMessage(content='You are a genZ poet.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Write a poem about the beauty of mountains.', additional_kwargs={}, response_metadata={})])

## Chain

In [12]:
# Step-1 : Create a prompt template
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a genZ poet."),
        ("user", "Write a poem about the {topic}.")
    ]
)

# Step-2 : Sending the prompt to the LLM
from pydantic import BaseModel, Field

class llm_schema(BaseModel):
    query: str = Field(description="The quesry that was asked to the LLM")
    answer: str = Field(description="The answer that the LLM generated")
    total_tokens: int = Field(description="The total number of tokens used in the LLM response")

llm_structured = llm.with_structured_output(llm_schema)

# Step-3 : Parsing the output
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=llm_schema)

# Step-4 : Putting all together

my_chain = my_template | llm_structured #| parser

In [13]:
my_chain.invoke({"topic": "beauty of mountains"})

llm_schema(query='Write a poem about the beauty of mountains.', answer="They rise like quiet gods, lowkey flexing against the sky,\npeaks iced in morning light, sunglasses for the sun.\nWind scrolls through cedar and rock, leaves whispering old playlists,\nclouds pause, double-tap the ridges, stunned.\n\nMy breath learns patience on the climb — slow, honest, real.\nAltitude teaches me how small my hurry was,\nhow loud the city’s notifications are compared to a glacier's pulse.\n\nAt the summit, the world folds open like a map you can finally read;\nvalleys deep as stories, rivers streaming like new messages.\nThe mountains hold secrets in their grooves, patient and unbothered.\n\nBeauty here is not flashy — it’s steady, ancient, a vibe that lingers.\nYou come up for the view and leave with a quieter heart,\na reminder that some things are built to last, silent and true.", total_tokens=140)